In [36]:
import logging
import os
from statistics import mean, stdev
import sys

import pandas as pd
import wandb
import torch
from pprint import pformat

from transformers import (
    AutoTokenizer,
    AutoConfig,
    HfArgumentParser,
    set_seed,
    TrainerCallback,
    EarlyStoppingCallback,
    Trainer,
    EvalPrediction,
    TrainingArguments,
)

from multimodal_exp_args import (
    MultimodalDataTrainingArguments,
    ModelArguments,
    OurTrainingArguments,
    ComputerStabilityArguments,
)
from evaluation import calc_stability_metrics
from multimodal_transformers.data import load_data_from_folder, load_data_into_folds
from multimodal_transformers.model import TabularConfig
from multimodal_transformers.model import AutoModelWithTabular
from multimodal_transformers.model import CustomTrainer
from util import create_dir_if_not_exists, get_args_info_as_str

In [37]:
from config.definitions import ROOT_DIR, RNDN

In [38]:
parser = HfArgumentParser(
    (ModelArguments, MultimodalDataTrainingArguments, OurTrainingArguments, ComputerStabilityArguments)
)
model_args, data_args, training_args, stability_args = parser.parse_json_file(
    json_file=os.path.abspath(ROOT_DIR + f"/datasets/processed/yelp/train_config.json")
)


# Setup logging
create_dir_if_not_exists(training_args.output_dir)
stream_handler = logging.StreamHandler(sys.stderr)
file_handler = logging.FileHandler(
    filename=os.path.join(training_args.output_dir, "train_log.txt"),  encoding='utf-8', mode="w+"
)
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s -   %(message)s",
    level=logging.INFO if training_args.local_rank in [-1, 0] else logging.WARN,
    datefmt="%m/%d/%Y %H:%M:%S",
    handlers=[stream_handler, file_handler],
)

tokenizer = AutoTokenizer.from_pretrained(
    model_args.tokenizer_name
    if model_args.tokenizer_name
    else model_args.model_name_or_path,
    cache_dir=model_args.cache_dir,
)

if not data_args.create_folds:
    train_dataset, val_dataset, test_dataset = load_data_from_folder(
        data_args.train_csv_name, 
        data_args.val_csv_name, 
        data_args.test_csv_name,
        data_args.num_random_samples,
        data_args.data_path,
        data_args.column_info["text_cols"],
        tokenizer,
        categorical_cols=data_args.column_info["cat_cols"],
        numerical_cols=data_args.column_info["num_cols"],
        categorical_encode_type=data_args.categorical_encode_type,
        numerical_transformer_method=data_args.numerical_transformer_method,
        sep_text_token_str=tokenizer.sep_token
        if not data_args.column_info["text_col_sep_token"]
        else data_args.column_info["text_col_sep_token"],
        max_token_length=training_args.max_token_length,
        debug=training_args.debug_dataset,
        debug_dataset_size=training_args.debug_dataset_size,
    )
    train_datasets = [train_dataset]
    val_datasets = [val_dataset]
    test_datasets = [test_dataset]
else:
    train_datasets, val_datasets, test_datasets = load_data_into_folds(
        data_args.data_path,
        data_args.num_folds,
        data_args.validation_ratio,
        data_args.column_info["text_cols"],
        tokenizer,
        categorical_cols=data_args.column_info["cat_cols"],
        numerical_cols=data_args.column_info["num_cols"],
        categorical_encode_type=data_args.categorical_encode_type,
        numerical_transformer_method=data_args.numerical_transformer_method,
        sep_text_token_str=tokenizer.sep_token
        if not data_args.column_info["text_col_sep_token"]
        else data_args.column_info["text_col_sep_token"],
        max_token_length=training_args.max_token_length,
        debug=training_args.debug_dataset,
        debug_dataset_size=training_args.debug_dataset_size,
    )
train_dataset = train_datasets[0]
set_seed(training_args.seed)

loading configuration file config.json from cache at C:\Users\FS-Ma/.cache\huggingface\hub\models--bert-base-multilingual-uncased\snapshots\3da6b6aad5111664db74322f2158b7f93e09a717\config.json
Model config BertConfig {
  "_name_or_path": "bert-base-multilingual-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.0",
  "type_vocab_size": 2,
  "use_cache": true,
  

In [41]:
total_results = []
for i, (train_dataset, val_dataset, test_dataset) in enumerate(
    zip(train_datasets, val_datasets, test_datasets)
):
    config = AutoConfig.from_pretrained(
        model_args.config_name
        if model_args.config_name
        else model_args.model_name_or_path,
        cache_dir=model_args.cache_dir,
    )
    tabular_config = TabularConfig(
        cat_feat_dim=train_dataset.cat_feats.shape[1]
        if train_dataset.cat_feats is not None
        else 0,
        numerical_feat_dim=train_dataset.numerical_feats.shape[1]
        if train_dataset.numerical_feats is not None
        else 0,
        **vars(data_args),
    )
    config.tabular_config = tabular_config
    model = AutoModelWithTabular.from_pretrained(
        model_args.config_name
        if model_args.config_name
        else model_args.model_name_or_path,
        config=config,
        cache_dir=model_args.cache_dir,
    )

loading configuration file config.json from cache at C:\Users\FS-Ma/.cache\huggingface\hub\models--bert-base-multilingual-uncased\snapshots\3da6b6aad5111664db74322f2158b7f93e09a717\config.json
Model config BertConfig {
  "_name_or_path": "bert-base-multilingual-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.0",
  "type_vocab_size": 2,
  "use_cache": true,
  

In [42]:
model.embeddings.requires_grad = False
model.encoder.requires_grad = False
model.pooler.requires_grad = False
model.dropout.requires_grad = False

In [46]:
trainer = CustomTrainer(
        model = model,
        args = training_args,
        train_dataset = train_dataset,
        eval_dataset = val_dataset,
        compute_metrics = None, #evaluation strategy to adopt during training calling by evaluation_strategy. See bertvaewithtabular/multimodal_transformers/model/custom_trainer.py#L261
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )
#trainer.add_callback(CustomCallback(trainer, stability_args, wandb))

In [48]:
trainer.train(
    model_path=model_args.model_name_or_path
    if os.path.isdir(model_args.model_name_or_path)
    else None
)

***** Running training *****
  Num examples = 10
  Num Epochs = 4
  Instantaneous batch size per device = 10
  Total train batch size (w. parallel, distributed & accumulation) = 10
  Gradient Accumulation steps = 1
  Total optimization steps = 4
  Number of trainable parameters = 169452230


  0%|          | 0/4 [00:00<?, ?it/s]

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 trainer.train(                                                                               │
│   2 │   model_path=model_args.model_name_or_path                                                 │
│   3 │   if os.path.isdir(model_args.model_name_or_path)                                          │
│   4 │   else None                                                                                │
│                                                                                                  │
│ c:\Users\FS-Ma\OneDrive\Documents\projects\customer-segmentation-analysis\lib\site-packages\tran │
│ sformers\trainer.py:1543 in train                                                                │
│                                                                                                  │
│   1540 │   │   inner_training_loop = find_executable_batch_size(                                 │
│   1541 │   │   │   self._inner_training_loop, self._train_batch_size, args.auto_find_batch_size  │
│   1542 │   │   )                                                                                 │
│ ❱ 1543 │   │   return inner_training_loop(                                                       │
│   1544 │   │   │   args=args,                                                                    │
│   1545 │   │   │   resume_from_checkpoint=resume_from_checkpoint,                                │
│   1546 │   │   │   trial=trial,                                                                  │
│                                                                                                  │
│ c:\Users\FS-Ma\OneDrive\Documents\projects\customer-segmentation-analysis\lib\site-packages\tran │
│ sformers\trainer.py:1791 in _inner_training_loop                                                 │
│                                                                                                  │
│   1788 │   │   │   │   │   with model.no_sync():                                                 │
│   1789 │   │   │   │   │   │   tr_loss_step = self.training_step(model, inputs)                  │
│   1790 │   │   │   │   else:                                                                     │
│ ❱ 1791 │   │   │   │   │   tr_loss_step = self.training_step(model, inputs)                      │
│   1792 │   │   │   │                                                                             │
│   1793 │   │   │   │   if (                                                                      │
│   1794 │   │   │   │   │   args.logging_nan_inf_filter                                           │
│                                                                                                  │
│ c:\Users\FS-Ma\OneDrive\Documents\projects\customer-segmentation-analysis\lib\site-packages\tran │
│ sformers\trainer.py:2557 in training_step                                                        │
│                                                                                                  │
│   2554 │   │   │   # loss gets scaled under gradient_accumulation_steps in deepspeed             │
│   2555 │   │   │   loss = self.deepspeed.backward(loss)                                          │
│   2556 │   │   else:                                                                             │
│ ❱ 2557 │   │   │   loss.backward()                                                               │
│   2558 │   │                                                                                     │
│   2559 │   │   return loss.detach()                                                              │
│   2560                                                     